# Batch PDF File-Based Extraction Evaluation

Evaluate PDF-based metadata extraction using OpenAI's native File API across open access Fuster validation samples.

**Approach:**
- Filter to open access PDFs only (via OpenAlex `is_oa` flag)
- Upload raw PDFs to OpenAI File API
- Use GPT-5-mini with native PDF understanding (text + visual analysis)
- Custom `PDF_SYSTEM_MESSAGE` optimized for document structure
- Compare against manually annotated ground truth
- Use `DatasetFeaturesNormalized` for ground truth validation

**Key Differences from Section-Based Pipeline:**
- No GROBID parsing required
- OpenAI processes both text AND images from each page
- Better for tables, figures, and visual content
- Higher token usage (text + image per page)

In [2]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project modules - using PDF file-based extraction
from llm_metadata.gpt_classify import (
    classify_pdf_file,
    classify_pdf_url,
    upload_pdf_to_openai,
    delete_openai_file,
    PDF_SYSTEM_MESSAGE,
)
from llm_metadata.openalex import get_work_by_doi, extract_pdf_url
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.schemas.evaluation import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# Paths
PDF_DIR = Path("data/pdfs/fuster")
MANIFEST_PATH = PDF_DIR / "manifest.csv"
GROUND_TRUTH_PATH = Path("data/dataset_092624.xlsx")
OUTPUT_DIR = Path("artifacts/pdf_file_results")

print(f"PDF directory: {PDF_DIR.absolute()}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
print(f"Ground truth exists: {GROUND_TRUTH_PATH.exists()}")

PDF directory: c:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest exists: True
Ground truth exists: True


## Step 1: Load Manifest and Ground Truth

Map dataset DOIs to article DOIs and load ground truth annotations.

In [3]:
# Load manifest with DOI mapping
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} total rows")

# Filter to downloaded PDFs only
downloaded_df = manifest_df[manifest_df['status'] == 'downloaded'].copy()
print(f"Downloaded PDFs: {len(downloaded_df)}")

# Create DOI mapping (dataset_doi -> article_doi)
doi_mapping = dict(zip(downloaded_df['dataset_doi'], downloaded_df['article_doi']))
print(f"DOI mappings: {len(doi_mapping)}")

# Display sample
downloaded_df[['article_doi', 'dataset_doi', 'downloaded_pdf_path']].head(5)

Manifest: 75 total rows
Downloaded PDFs: 70
DOI mappings: 70


,article_doi,dataset_doi,downloaded_pdf_path
1,10.1093/jhered/esx103,10.5061/dryad.121sk,fuster\10.1093_jhered_esx103.pdf
2,10.1371/journal.pone.0128238,10.5061/dryad.1771t,fuster\10.1371_journal.pone.0128238.pdf
3,10.1371/journal.pone.0073695,10.5061/dryad.1cc4v,fuster\10.1371_journal.pone.0073695.pdf
4,10.1002/ece3.4685,10.5061/dryad.24q6q70,fuster\10.1002_ece3.4685.pdf
5,10.1639/0007-2745-119.1.008,10.5061/dryad.24rj8,fuster\10.1639_0007-2745-119.1.008.pdf


In [4]:
# Load ground truth
gt_df = pd.read_excel(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} total rows")

# Extract dataset DOI from URL column
gt_df['dataset_doi'] = gt_df['url'].str.replace('https://doi.org/', '', regex=False)

# Filter to DOIs that have downloaded PDFs
gt_matched = gt_df[gt_df['dataset_doi'].isin(doi_mapping.keys())].copy()
print(f"Ground truth with matched PDFs: {len(gt_matched)}")

# Add article DOI mapping
gt_matched['article_doi'] = gt_matched['dataset_doi'].map(doi_mapping)

gt_matched[['dataset_doi', 'article_doi', 'data_type', 'species']].head(5)

Ground truth: 418 total rows
Ground truth with matched PDFs: 70


,dataset_doi,article_doi,data_type,species
2,10.5061/dryad.121sk,10.1093/jhered/esx103,EBV genetic analysis,Glyptemys insculpta
4,10.5061/dryad.1771t,10.1371/journal.pone.0128238,density,"raccoons, striped skunks"
6,10.5061/dryad.1cc4v,10.1371/journal.pone.0073695,presence only,Rangifer tarandus caribou
7,10.5061/dryad.24q6q70,10.1002/ece3.4685,other,Rangifer tarandus caribou
8,10.5061/dryad.24rj8,10.1639/0007-2745-119.1.008,genetic analysis,Aspicilia bicensis


## Step 2: Filter to Open Access PDFs Only

Query OpenAlex to identify open access papers and get PDF URLs where available.

In [5]:
# Query OpenAlex for open access status
from time import sleep

openalex_data = []
print("Querying OpenAlex for open access status...")

for idx, row in gt_matched.iterrows():
    article_doi = row['article_doi']
    dataset_doi = row['dataset_doi']
    
    try:
        work = get_work_by_doi(article_doi)
        if work:
            is_oa = work.get('open_access', {}).get('is_oa', False)
            oa_status = work.get('open_access', {}).get('oa_status', 'unknown')
            pdf_url = extract_pdf_url(work)
            
            openalex_data.append({
                'dataset_doi': dataset_doi,
                'article_doi': article_doi,
                'is_oa': is_oa,
                'oa_status': oa_status,
                'pdf_url': pdf_url,
            })
        else:
            openalex_data.append({
                'dataset_doi': dataset_doi,
                'article_doi': article_doi,
                'is_oa': False,
                'oa_status': 'not_found',
                'pdf_url': None,
            })
    except Exception as e:
        print(f"Error querying {article_doi}: {e}")
        openalex_data.append({
            'dataset_doi': dataset_doi,
            'article_doi': article_doi,
            'is_oa': False,
            'oa_status': 'error',
            'pdf_url': None,
        })
    
    # Rate limit to avoid hitting OpenAlex
    sleep(0.1)

openalex_df = pd.DataFrame(openalex_data)
print(f"\nOpenAlex query complete: {len(openalex_df)} records")

Querying OpenAlex for open access status...

OpenAlex query complete: 70 records


In [6]:
# Filter to open access only
oa_df = openalex_df[openalex_df['is_oa'] == True].copy()
print(f"Open access papers: {len(oa_df)} out of {len(openalex_df)}")

# Show OA status breakdown
print("\nOpen Access Status Breakdown:")
print(openalex_df['oa_status'].value_counts())

# Show how many have direct PDF URLs
has_pdf_url = oa_df['pdf_url'].notna().sum()
print(f"\nOA papers with direct PDF URL: {has_pdf_url}")
print(f"OA papers requiring local file: {len(oa_df) - has_pdf_url}")

Open access papers: 44 out of 70

Open Access Status Breakdown:
oa_status
gold         25
closed       19
bronze       17
not_found     7
green         2
Name: count, dtype: int64

OA papers with direct PDF URL: 39
OA papers requiring local file: 5


In [7]:
# Validate ground truth with DatasetFeaturesNormalized (OA papers only)
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset'
]

# Filter ground truth to OA papers only
oa_dataset_dois = set(oa_df['dataset_doi'])
gt_oa = gt_matched[gt_matched['dataset_doi'].isin(oa_dataset_dois)].copy()

ground_truth_by_dataset = {}
validation_errors = []

for _, row in gt_oa.iterrows():
    dataset_doi = row['dataset_doi']
    try:
        # Extract relevant columns and convert to dict
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        
        # Validate using DatasetFeaturesNormalized
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_dataset[dataset_doi] = validated
    except Exception as e:
        validation_errors.append({'doi': dataset_doi, 'error': str(e)})

print(f"Successfully validated: {len(ground_truth_by_dataset)} records")
print(f"Validation errors: {len(validation_errors)} records")

if validation_errors:
    print("\nFirst 3 errors:")
    for err in validation_errors[:3]:
        print(f"  {err['doi']}: {err['error'][:100]}...")

Successfully validated: 44 records
Validation errors: 0 records


## Step 3: Configure PDF File Pipeline

Set up PDF file-based extraction with OpenAI's native PDF support and custom system prompt.

In [8]:
# Pipeline configuration
MODEL = "gpt-5-mini"
REASONING = {"effort": "low"}
MAX_OUTPUT_TOKENS = 4096

print("PDF File Pipeline Configuration:")
print(f"  Model: {MODEL}")
print(f"  Reasoning: {REASONING}")
print(f"  Max output tokens: {MAX_OUTPUT_TOKENS}")
print(f"  Extraction schema: DatasetFeatures")
print(f"\nSystem Prompt Preview (first 300 chars):")
print(f"  {PDF_SYSTEM_MESSAGE[:300]}...")

PDF File Pipeline Configuration:
  Model: gpt-5-mini
  Reasoning: {'effort': 'low'}
  Max output tokens: 4096
  Extraction schema: DatasetFeatures

System Prompt Preview (first 300 chars):
  You are EcodataGPT, a structured data extraction engine for scientific PDF analysis.

Goal: Extract biodiversity dataset features from the provided PDF document into the provided schema.

## PDF Document Analysis

This PDF has been processed to provide both:
1. **Extracted text** from each page for ...


## Step 4: Run PDF File-Based Extraction

Process all open access PDFs through OpenAI's File API.

In [9]:
# Prepare records for processing
# Merge OA data with manifest to get PDF paths
oa_with_paths = oa_df.merge(
    downloaded_df[['dataset_doi', 'downloaded_pdf_path']],
    on='dataset_doi',
    how='left'
)

# Build processing records
processing_records = []
for _, row in oa_with_paths.iterrows():
    dataset_doi = row['dataset_doi']
    pdf_path_rel = row['downloaded_pdf_path']
    pdf_url = row['pdf_url']
    
    # Construct full PDF path
    pdf_path = PDF_DIR / pdf_path_rel if pdf_path_rel else None
    if pdf_path and not pdf_path.exists():
        pdf_path = PDF_DIR.parent / pdf_path_rel
    
    if pdf_path and pdf_path.exists():
        processing_records.append({
            'dataset_doi': dataset_doi,
            'article_doi': row['article_doi'],
            'pdf_path': str(pdf_path),
            'pdf_url': pdf_url,
            'use_url': False # pdf_url is not None,  # Prefer URL if available
        })

print(f"Records to process: {len(processing_records)}")
print(f"  - With direct URL: {sum(1 for r in processing_records if r['use_url'])}")
print(f"  - Local file upload: {sum(1 for r in processing_records if not r['use_url'])}")

Records to process: 44
  - With direct URL: 0
  - Local file upload: 44


In [ ]:
# Run extraction
from dataclasses import dataclass
from typing import Optional, Any

@dataclass
class PDFExtractionResult:
    dataset_doi: str
    article_doi: str
    status: str
    extraction: Optional[dict] = None
    error_message: Optional[str] = None
    input_tokens: Optional[int] = None
    output_tokens: Optional[int] = None
    cost_usd: Optional[float] = None
    file_id: Optional[str] = None
    extraction_method: Optional[str] = None

results = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running PDF file-based extraction on {len(processing_records)} papers...")
print(f"Using {MODEL} with {REASONING}")
print()

for i, record in enumerate(processing_records):
    dataset_doi = record['dataset_doi']
    print(f"[{i+1}/{len(processing_records)}] Processing {dataset_doi}...", end=" ")
    
    try:
        if record['use_url'] and record['pdf_url']:
            # Use URL-based extraction
            result = classify_pdf_url(
                pdf_url=record['pdf_url'],
                system_message=PDF_SYSTEM_MESSAGE,
                model=MODEL,
                reasoning=REASONING,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                text_format=DatasetFeatures,
            )
        else:
            # Use file upload extraction
            result = classify_pdf_file(
                pdf_path=record['pdf_path'],
                system_message=PDF_SYSTEM_MESSAGE,
                model=MODEL,
                reasoning=REASONING,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                text_format=DatasetFeatures,
            )
        
        # Extract usage stats
        usage_cost = result.get('usage_cost', {})
        
        results.append(PDFExtractionResult(
            dataset_doi=dataset_doi,
            article_doi=record['article_doi'],
            status="success",
            extraction=result['output'].model_dump(mode="python") if result.get('output') else None,
            input_tokens=usage_cost.get('input_tokens'),
            output_tokens=usage_cost.get('output_tokens'),
            cost_usd=usage_cost.get('total_cost'),
            file_id=result.get('file_id'),
            extraction_method=result.get('extraction_method'),
        ))
        print(f"OK (${usage_cost.get('total_cost', 0):.4f})")
        
    except Exception as e:
        results.append(PDFExtractionResult(
            dataset_doi=dataset_doi,
            article_doi=record['article_doi'],
            status="error",
            error_message=str(e),
        ))
        print(f"ERROR: {str(e)[:50]}")

# Summary
success_count = sum(1 for r in results if r.status == "success")
error_count = sum(1 for r in results if r.status == "error")
print(f"\nProcessing complete: {success_count} success, {error_count} errors")

NameError: name 'datetime' is not defined

In [ ]:
# Save results to manifest
results_df = pd.DataFrame([vars(r) for r in results])
output_manifest_path = OUTPUT_DIR / f"pdf_file_results_{timestamp}.csv"
results_df.to_csv(output_manifest_path, index=False)
print(f"Results saved to: {output_manifest_path}")

# Show extraction method breakdown
print("\nExtraction method breakdown:")
print(results_df['extraction_method'].value_counts())

Results saved to: artifacts\pdf_file_results\pdf_file_results_20260115_093041.csv

Extraction method breakdown:
extraction_method
openai_url_api    8
Name: count, dtype: int64


## Step 5: Prepare Extractions for Evaluation

Convert extraction results to Pydantic models for evaluation.

In [ ]:
# Build predictions by dataset DOI
predictions_by_dataset = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    
    dataset_doi = result.dataset_doi
    if not dataset_doi:
        continue
    
    try:
        # Create DatasetFeatures from extraction dict
        pred = DatasetFeatures.model_validate(result.extraction)
        predictions_by_dataset[dataset_doi] = pred
    except Exception as e:
        print(f"Error validating prediction for {dataset_doi}: {e}")

print(f"Valid predictions: {len(predictions_by_dataset)}")

# Find common DOIs between predictions and ground truth
common_dois = set(predictions_by_dataset.keys()) & set(ground_truth_by_dataset.keys())
print(f"Common DOIs for evaluation: {len(common_dois)}")

Valid predictions: 8
Common DOIs for evaluation: 8


## Step 6: Evaluate PDF File-Based Extraction

Compare PDF file-based extractions against ground truth using evaluation framework.

In [ ]:
# Evaluation fields
EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f', 'species'
]

# Evaluation configuration with fuzzy matching for species
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    fuzzy_match_fields={
        'species': FuzzyMatchConfig(threshold=80),
    }
)

# Filter to common DOIs
true_by_id = {doi: ground_truth_by_dataset[doi] for doi in common_dois}
pred_by_id = {doi: predictions_by_dataset[doi] for doi in common_dois}

# Run evaluation
pdf_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("PDF FILE-BASED Extraction Metrics:")
print("=" * 70)
display(pdf_report.metrics_to_pandas())

PDF FILE-BASED Extraction Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,5,23,3,0,8,0.178571,0.625000,0.277778,0.625,0.000
1,geospatial_info_dataset,2,33,1,0,8,0.057143,0.666667,0.105263,0.250,0.000
2,spatial_range_km2,3,2,2,3,8,0.600000,0.600000,0.600000,0.750,0.750
6,species,1,54,16,0,8,0.018182,0.058824,0.027778,0.125,0.000
5,temp_range_f,7,1,0,0,8,0.875000,1.000000,0.933333,0.875,0.875
4,temp_range_i,7,1,0,0,8,0.875000,1.000000,0.933333,0.875,0.875
3,temporal_range,0,8,7,0,8,0.000000,0.000000,NaN,0.000,0.000


In [ ]:
# Aggregate metrics
pdf_micro = micro_average(pdf_report.field_metrics.values())
pdf_macro = macro_f1(pdf_report.field_metrics.values())

print("\nAggregate Metrics:")
print("=" * 50)
print(f"{'Metric':<25} {'Value':>15}")
print("-" * 50)
print(f"{'Micro Precision':<25} {pdf_micro.precision or 0:>15.3f}")
print(f"{'Micro Recall':<25} {pdf_micro.recall or 0:>15.3f}")
print(f"{'Micro F1':<25} {pdf_micro.f1 or 0:>15.3f}")
print(f"{'Macro F1':<25} {pdf_macro or 0:>15.3f}")
print(f"{'Records Evaluated':<25} {len(common_dois):>15}")
print("=" * 50)


Aggregate Metrics:
Metric                              Value
--------------------------------------------------
Micro Precision                     0.170
Micro Recall                        0.463
Micro F1                            0.249
Macro F1                            0.480
Records Evaluated                       8


## Step 7: Per-Field Analysis

In [ ]:
# Per-field metrics for PDF file-based extraction
field_data = []

for field in EVAL_FIELDS:
    fm = pdf_report.field_metrics.get(field)
    if fm:
        field_data.append({
            'field': field,
            'precision': round(fm.precision or 0, 3),
            'recall': round(fm.recall or 0, 3),
            'f1': round(fm.f1 or 0, 3),
            'tp': fm.tp,
            'fp': fm.fp,
            'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics:")
display(field_df)

# Identify best and worst performing fields
if len(field_data) > 0:
    best = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest performing field: {best['field']} (F1={best['f1']})")
    print(f"Worst performing field: {worst['field']} (F1={worst['f1']})")

Per-Field Metrics:


,field,precision,recall,f1,tp,fp,fn,exact_match_rate
0,data_type,0.179,0.625,0.278,5,23,3,0.000
1,geospatial_info_dataset,0.057,0.667,0.105,2,33,1,0.000
2,spatial_range_km2,0.600,0.600,0.600,3,2,2,0.750
3,temporal_range,0.000,0.000,0.000,0,8,7,0.000
4,temp_range_i,0.875,1.000,0.933,7,1,0,0.875
5,temp_range_f,0.875,1.000,0.933,7,1,0,0.875
6,species,0.018,0.059,0.028,1,54,16,0.000



Best performing field: temp_range_i (F1=0.933)
Worst performing field: temporal_range (F1=0)


In [ ]:
# Detailed per-record comparison results
print("Per-record, per-field results (mismatches highlighted):")
results_detail_df = pdf_report.to_pandas()
mismatches = results_detail_df[~results_detail_df['match']]
print(f"\nTotal mismatches: {len(mismatches)}")
display(mismatches)

Per-record, per-field results (mismatches highlighted):

Total mismatches: 36


,record_id,field,true_value,pred_value,match,tp,fp,fn,tn
0,10.5061/dryad.1771t,data_type,[density],"[abundance, density, distribution]",False,1,2,0,0
1,10.5061/dryad.1771t,geospatial_info_dataset,[sample],"[sample, site, site_ids, maps, administrative_...",False,1,5,0,0
3,10.5061/dryad.1771t,temporal_range,2008,27 September to 20 October 2008,False,0,1,1,0
6,10.5061/dryad.1771t,species,"[raccoons, striped skunks]",[Procyon lotor (raccoon) — 610 individuals cap...,False,0,2,2,0
7,10.5061/dryad.1cc4v,data_type,[presence-only],"[presence-only, distribution, time_series]",False,1,2,0,0
8,10.5061/dryad.1cc4v,geospatial_info_dataset,None,"[sample, site, range, maps, administrative_units]",False,0,5,0,0
10,10.5061/dryad.1cc4v,temporal_range,"1999-2000, 2004-2011",1999–2000 and 2004–2011 (VHF 1999–2000; GPS 20...,False,0,1,1,0
13,10.5061/dryad.1cc4v,species,[Rangifer tarandus caribou],[forest-dwelling woodland caribou (Rangifer ta...,False,0,3,1,0
14,10.5061/dryad.679s1dt,data_type,[presence-absence],"[abundance, presence-absence, species_richness...",False,1,4,0,0
15,10.5061/dryad.679s1dt,geospatial_info_dataset,[site_ids],"[site, maps, administrative_units]",False,0,3,1,0


## Step 8: Cost Analysis

In [ ]:
# Cost analysis from results
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens = sum(r.output_tokens or 0 for r in successful_results)
    
    print("\nCOST ANALYSIS (PDF File-Based Extraction)")
    print("=" * 50)
    print(f"{'Metric':<30} {'Value':>18}")
    print("-" * 50)
    print(f"{'Total PDFs Processed':<30} {len(successful_results):>18}")
    print(f"{'Avg Input Tokens per PDF':<30} {total_input_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Output Tokens per PDF':<30} {total_output_tokens / len(successful_results):>18,.0f}")
    print("-" * 50)
    print(f"{'Total Input Tokens':<30} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<30} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<30} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per PDF (USD)':<30} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 50)
    
    # Method breakdown
    url_results = [r for r in successful_results if r.extraction_method == 'openai_url_api']
    file_results = [r for r in successful_results if r.extraction_method == 'openai_file_api']
    
    if url_results:
        url_cost = sum(r.cost_usd or 0 for r in url_results)
        print(f"\nURL-based extraction: {len(url_results)} papers, ${url_cost:.4f} total")
    
    if file_results:
        file_cost = sum(r.cost_usd or 0 for r in file_results)
        print(f"File upload extraction: {len(file_results)} papers, ${file_cost:.4f} total")


COST ANALYSIS (PDF File-Based Extraction)
Metric                                      Value
--------------------------------------------------
Total PDFs Processed                            8
Avg Input Tokens per PDF                   30,357
Avg Output Tokens per PDF                   2,147
--------------------------------------------------
Total Input Tokens                        242,858
Total Output Tokens                        17,179
Total Cost (USD)               $           0.1247
Avg Cost per PDF (USD)         $          0.01559

URL-based extraction: 8 papers, $0.1247 total


## Summary

This notebook evaluated PDF file-based metadata extraction using OpenAI's native PDF understanding:

**Key Features:**
- Used OpenAI File API for raw PDF upload
- Native PDF processing (text + visual analysis per page)
- Custom `PDF_SYSTEM_MESSAGE` optimized for document structure
- Filtered to open access papers only

**Comparison with Section-Based Pipeline:**
- No GROBID parsing required (simpler setup)
- Better handling of tables, figures, and visual content
- Higher token usage (includes image encoding)
- More expensive per document

**Next Steps:**
- Compare metrics with section-based extraction
- Analyze which fields benefit from visual analysis
- Optimize cost vs. accuracy tradeoff